# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dushyant1607/ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions
Finding 1: Content Age and Growth

Paper finding: The FlyRank research paper reports that growing content averaged 3,180 words and 184 days old, while declining content averaged 2,311 words and 230 days old. The paper describes this as an observational comparison, not proof of causation.

Methodology question: Were topic, search demand, and starting search visibility controlled for? Could these factors partly explain the differences between growing and declining content?

Finding 2: Content Freshness

Paper finding: The paper reports stronger performance for recently refreshed content and concludes that freshness can amplify existing content quality rather than replace it.

Methodology question: Were refreshed pages compared with similar unrefreshed pages? How did the study account for differences in prior performance and content quality when interpreting the relationship between refreshing and outcomes?

Reflection

These findings provide useful patterns for prioritizing content reviews, but observational comparisons alone cannot establish that content length, age, or refreshing directly causes better search performance. A stronger validation design would compare appropriately matched pages and clearly document how outcomes were measured.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

y = (df["trend_direction"] == "down").astype(int)

exclude = {
    "content_id", "client_id", "trend_direction", "trend_pct"
}
exclude.update(c for c in df.columns
               if c.endswith("_last_30d") or c.endswith("_prev_30d"))
exclude.update({
    "age_tier", "age_tier_order", "freshness_tier",
    "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier"
})

X = df.drop(columns=list(exclude), errors="ignore")
X = X.drop(columns=["is_declining_label"], errors="ignore")

# Remove rows with missing target, if any
valid = y.notna()
X, y, groups = X.loc[valid], y.loc[valid], df.loc[valid, "client_id"]

def evaluate(split_name, train_idx, test_idx):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns
    num_cols = X_train.select_dtypes(include="number").columns

    preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median",
                                     add_indicator=True)),
            ("scale", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ])

    model = Pipeline([
        ("preprocess", preprocess),
        ("classifier", LogisticRegression(
            class_weight="balanced", max_iter=1000, random_state=42
        ))
    ])
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]

    # Same top-50 evaluation rule for each split
    top_n = min(50, len(y_test))
    top_idx = probabilities.argsort()[-top_n:]
    precision_at_50 = y_test.iloc[top_idx].mean()

    return {
        "split": split_name,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "test_decline_rate": round(y_test.mean(), 4),
        "precision_at_50": round(precision_at_50, 4)
    }

# Before: random row split
train_idx, test_idx = train_test_split(
    range(len(X)), test_size=0.20, random_state=42,
    stratify=y
)
random_result = evaluate("Random row split", train_idx, test_idx)

# After: hold out entire clients
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
group_result = evaluate("Client-grouped split", train_idx, test_idx)

comparison = pd.DataFrame([random_result, group_result])
display(comparison)



,split,train_rows,test_rows,test_decline_rate,precision_at_50
0,Random row split,24000,6000,0.542,0.82
1,Client-grouped split,23837,6163,0.511,0.74


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
# Audit features that could leak target information
audit = pd.DataFrame({
    "feature": df.columns,
    "included_in_model": [
        c in X.columns for c in df.columns
    ]
})

audit["risk"] = audit["feature"].apply(
    lambda c: (
        "DIRECT TARGET LEAKAGE"
        if c in ["trend_direction", "trend_pct", "is_declining_label"]
        else "TEMPORAL OVERLAP — EXCLUDED"
        if c.endswith("_last_30d") or c.endswith("_prev_30d")
        else "IDENTIFIER — EXCLUDED"
        if c in ["content_id", "client_id"]
        else "REVIEW"
    )
)

display(audit[audit["risk"] != "REVIEW"])


,feature,included_in_model,risk
0,content_id,False,IDENTIFIER — EXCLUDED
1,client_id,False,IDENTIFIER — EXCLUDED
22,impressions_last_30d,False,TEMPORAL OVERLAP — EXCLUDED
23,clicks_last_30d,False,TEMPORAL OVERLAP — EXCLUDED
24,sessions_last_30d,False,TEMPORAL OVERLAP — EXCLUDED
25,impressions_prev_30d,False,TEMPORAL OVERLAP — EXCLUDED
26,clicks_prev_30d,False,TEMPORAL OVERLAP — EXCLUDED
27,sessions_prev_30d,False,TEMPORAL OVERLAP — EXCLUDED
42,trend_direction,False,DIRECT TARGET LEAKAGE
43,trend_pct,False,DIRECT TARGET LEAKAGE


trend_direction defines the target and must not be a feature.



trend_pct is derived from the trend label and must be excluded.



Recent and previous 30-day metrics overlap with the period used to define the trend, so they were excluded.



content_id and client_id are identifiers, not predictive features. client_id is used only to group the validation split.



## 4. Claim rewrite

### Original claim
The model can reliably identify declining content and prioritize pages for SEO improvements.

### Revised claim
In this dataset, the Logistic Regression model achieved Precision@50 of 82% with a random row split and 74% with a client-grouped split. The lower score on unseen clients suggests that random-split performance may overstate generalization. These results are preliminary and do not establish performance on future data or prove that acting on the predictions improves SEO outcomes.

### Limitation
The target labels are derived from 30-day impression trends, so the model predicts the dataset's defined decline category rather than independently verified future decline. Validation on a later time period and evaluation of real-world outcomes would be needed before operational use.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.